# Waypoint 기반 차량 주행
Vehicle Controller를 직접 작성하여 조작하는 방식의 한계를 극복하고, 더 정교한 시나리오를 구하기 위해 Waypoint 기반 제어 방법을 사용하였습니다.

Waypoint 기반 제어의 핵심은 CARLA가 제공하는 Waypoint와 PID 컨트롤러를 활용하여 차량이 차선의 중심을 안정적으로 따라가도록 자동 제어하는 방식입니다. 이 방식은 다음의 스텝을 따릅니다.
- 경로 탐색 : 현재 차량 위치에서 가장 가까운 도로 위의 Waypoint를 찾습니다.
- 목표 설정 : 탐색한 Waypoint로부터 일정 거리 앞에 있는 다음 Waypoint를 주행 목표로 설정합니다.
- 자동 제어 : CARLA의 PID 컨트롤러가 목표 Waypoint까지 도달하기 위한 최적의 throttle, brake steer 값을 계산합니다.
- 주행 : PID 컨트롤러가 계산한 제어 값을 차량에 적용하여 목표 지점으로 주행합니다.
- 반복 : 위 과정을 지속 반복하여 경로를 이탈하지 않고 안정적인 주행을 유지합니다.

In [12]:
import carla
import sys
import math
from queue import Queue

CARLA_AGENT_PATH = '/home/oem/workspace/carla_workshop/carla_0.9.14/PythonAPI/carla/'
sys.path.append(CARLA_AGENT_PATH)
from agents.navigation import controller as ctr


class DriveWaypoint(object):
    """
    상태 머신(State Machine) 기반의 안전한 Waypoint 추종 클래스
    """
    def __init__(self, vehicle, pid_controller, world):
        self.car = vehicle
        self.pid = pid_controller
        self.world = world
        self.dir_wp_que = Queue()
        self.is_stopping = False # 차량의 현재 주행 상태를 관리하는 플래그
        
        # '현재 위치'가 아닌 추종해야 할 '목표 웨이포인트'를 별도로 관리
        self.target_waypoint = self.world.get_map().get_waypoint(self.car.get_location())

    def drive_step(self, speed=30.0, debug=False):

        if self.is_stopping:
            control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
            self.car.apply_control(control)
            return
        
        current_loc = self.car.get_location()
        
        # 차량의 현재 실제 속도(m/s)를 계산
        velocity = self.car.get_velocity()
        current_speed_ms = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
        
        # 속도에 비례하는 동적 주시 거리(Look-ahead) 및 갱신 반경 계산
        # 시속 0일때 최소 3m, 속도가 빠를수록 거리가 멀어짐 (예: 시속 72km(20m/s)일 때 약 13m 앞을 봄)
        dynamic_lookahead = max(3.0, current_speed_ms * 0.5 + 3.0) 
        
        # 목표 도달 판정 거리도 속도에 비례해 넓혀줌 (고속일수록 목표점을 넉넉히 스쳐지나가도 도달한 것으로 인정)
        reach_threshold = max(1.5, current_speed_ms * 0.2)

        dist_to_target = current_loc.distance(self.target_waypoint.transform.location)

        if dist_to_target <= reach_threshold: # 목표점과의 거리가 도달
            if not self.dir_wp_que.empty():
                self.target_waypoint = self.dir_wp_que.get()
                print("[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨")
            else:
                # 하드코딩된 3.0 대신 동적 거리 적용
                next_wps = self.target_waypoint.next(dynamic_lookahead)
                if next_wps: 
                    self.target_waypoint = next_wps[0]

        if debug:
            self.world.debug.draw_point(
                self.target_waypoint.transform.location, 
                size=0.15, 
                color=carla.Color(255, 0, 0), # 고속 주행 시 목표점이 멀리 찍히는 것을 빨간 점으로 확인
                life_time=0.1
            )

        control = self.pid.run_step(target_speed=speed, waypoint=self.target_waypoint)
        self.car.apply_control(control)

    def add_left_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 좌측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            left_wp = next_wps[0].get_left_lane()
            # [핵심 방어 로직] 좌측 차선이 실제로 존재하는지 검증
            if left_wp is not None:
                self.dir_wp_que.put(left_wp)
                print(f"[명령] {distance}m 앞 좌측 차선 변경 예약 완료")
            else:
                print("[경고] 좌측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def add_right_waypoint(self, distance=20):
        """
        현재 위치에서 distance(m) 앞의 우측 차선 웨이포인트를 큐에 추가
        """
        current_wp = self.world.get_map().get_waypoint(self.car.get_location())
        next_wps = current_wp.next(distance)
        
        if next_wps:
            right_wp = next_wps[0].get_right_lane()
            # [핵심 방어 로직] 우측 차선이 실제로 존재하는지 검증
            if right_wp is not None:
                self.dir_wp_que.put(right_wp)
                print(f"[명령] {distance}m 앞 우측 차선 변경 예약 완료")
            else:
                print("[경고] 우측에 유효한 차선이 없습니다. 차선 변경 명령 무시됨.")

    def stop(self):
        self.is_stopping = True
        print("[명령] 차량 정지 명령이 적용되었습니다.")
    
    def resume(self):
        self.is_stopping = False
        print("[명령] 차량 주행 재개 명령이 적용되었습니다.")  
        

In [27]:
import time
import carla

def run_lane_change_scenario():
    # 1. 클라이언트 연결 및 동기 모드 강제 설정 (커널 크래시 방지)
    client = carla.Client('localhost', 2000)
    client.set_timeout(10.0)
    world = client.get_world()

    if 'Town04' not in world.get_map().name :
        client.load_world('Town04')
        print(f"[시스템] 맵 '{world.get_map().name}' 로드 완료.")
    
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.02 # 50 FPS 고정
    world.apply_settings(settings)

    

    actor_list = []

    try:
        # 2. Ego Vehicle 스폰
        blueprint_library = world.get_blueprint_library()
        vehicle_bp = blueprint_library.filter('vehicle.audi.tt')[0]
        vehicle_bp.set_attribute('role_name', 'hero')

        # ego vehicle(중심 차량)
        vehicle_bp_2 = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
        vehicle_bp_2.set_attribute('color','255,255,255')
        
        
        # spawn_point = world.get_map().get_spawn_points()[0]
        approximate_location = carla.Location(x=160, y=14,z=9)
        approximate_location_A = carla.Location(x=146.1, y=16, z=11.4)
        spawn_point = world.get_map().get_waypoint(approximate_location).transform
        spawn_point_A = world.get_map().get_waypoint(approximate_location_A).transform
        spawn_point.location.z += 0.5 # 물리 엔진 충돌 방지용 Z축 보정
        
        spawn_point_A.location.z += 0.5 # 물리 엔진 충돌 방지용 Z축 보정
        ego_vehicle = world.spawn_actor(vehicle_bp, spawn_point)
        car_A = world.spawn_actor(vehicle_bp_2, spawn_point_A)
        actor_list.append(ego_vehicle)
        actor_list.append(car_A)

        print("\n[시스템] 차량 스폰 완료. 물리 엔진 안착을 위해 1초 대기합니다...")

        brake_control = carla.VehicleControl(throttle=0.0, steer=0.0, brake=1.0)
        ego_vehicle.apply_control(brake_control)

        for _ in range(20): # 20틱 * 0.05초 = 0.5초 대기
            world.tick()
            time.sleep(0.05)
        
        print(f"[시스템] 차량 스폰 완료. 시나리오를 시작합니다.")

        # 3. 제어기 및 DriveWaypoint 객체 단일 초기화
        args_lateral_dict = {'K_P': 0.8, 'K_I': 0.05, 'K_D': 0.1, 'dt': 0.05} # 고속 주행 시 안정성을 위한 튜닝
        args_long_dict = {'K_P': 1.0, 'K_I': 0.05, 'K_D': 0.0, 'dt': 0.05}
        
        pid_controller = ctr.VehiclePIDController(ego_vehicle, args_lateral_dict, args_long_dict)
        driver = DriveWaypoint(ego_vehicle, pid_controller, world)

        pid_controller_A = ctr.VehiclePIDController(car_A, args_lateral_dict, args_long_dict)
        driver_A = DriveWaypoint(car_A, pid_controller_A, world)

        # 4. 메인 제어 루프 (약 10초 주행 = 500 틱 * 0.02초)
        for tick in range(700):
            world.tick() # 서버 프레임 동기화
            time.sleep(0.02)
            
            # [시나리오 트리거] 특정 틱에 도달 시 차선 변경 명령 큐(Queue)에 주입
            if tick == 150: # 3초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 우측 차선 변경 명령 하달")
                driver.add_right_waypoint(distance=20)
            elif tick == 350: # 7초 경과 시점
                print(f"\n[Tick {tick}] 이벤트 발생: 좌측 차선 변경 (복귀) 명령 하달")
                driver.add_left_waypoint(distance=20)
            
            # 단일 스텝 주행 실행 (디버그 라인 렌더링 활성화)
            driver.drive_step(speed=90.0, debug=True)
            driver_A.drive_step(speed=35.0, debug=True) 

    finally:
        # 5. 환경 초기화 (비정상 종료 시에도 반드시 실행되어야 하는 방어 로직)
        print("\n[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.")
        settings = world.get_settings()
        settings.synchronous_mode = False
        world.apply_settings(settings)
        client.apply_batch([carla.command.DestroyActor(x) for x in actor_list])
        print("[시스템] 정리 완료.")


In [ ]:
# 코드 실행
run_lane_change_scenario()


[시스템] 차량 스폰 완료. 물리 엔진 안착을 위해 1초 대기합니다...
[시스템] 차량 스폰 완료. 시나리오를 시작합니다.

[Tick 150] 이벤트 발생: 우측 차선 변경 명령 하달
[명령] 20m 앞 우측 차선 변경 예약 완료
[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨

[Tick 350] 이벤트 발생: 좌측 차선 변경 (복귀) 명령 하달
[명령] 20m 앞 좌측 차선 변경 예약 완료
[시스템] 큐(Queue)에서 새 이벤트 웨이포인트 적용됨

[시스템] 시나리오 종료. 할당된 Actor 및 환경을 초기화합니다.
[시스템] 정리 완료.


: 